In [1]:
import torch
import torch_geometric
import lightning as L
from rdkit.Chem import rdmolfiles
from torch_geometric.utils.smiles import from_rdmol
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool

import torch.nn.functional as F
from torch import nn
from torch import optim

import pandas as pd

from data_processing.common.ids import canonicalize_smiles
from pathlib import Path
import re


from models.common.lightning_callbacks import EpochLossHistoryCallback

/Users/zanechan/anaconda3/envs/pyg_pose/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Workflow:
- Create toy dataset.
    - This dataset should be pathologically easy. 
    - Maybe just like 10 graphs
- Test real model architecture on the toy dataset
- Train that model. 

# Create Toy Dataset
Loading a mini testing dataset of pytorch graphs to test the model training pipeline without lmdb pain

In [2]:
VARIANT_SUFFIX_RE = re.compile(r"^(?P<ligand>.+)-(?P<variant_idx>\d+)$")

def _parse_variant_ligand(variant: str) -> str:
    """Strip the random suffix from `s_lp_Variant` and canonicalize the ligand."""
    variant = str(variant).strip()
    match = VARIANT_SUFFIX_RE.match(variant)
    if not match:
        raise ValueError(
            "Expected s_lp_Variant to look like '{smiles}-{int}', "
            f"got: {variant}"
        )
    return canonicalize_smiles(match.group("ligand"))

def _iter_ligand_mols(sdf_path: Path):
    """Yield `(mol_index, mol)` for valid ligand molecules in a docked SDF."""
    supplier = rdmolfiles.SDMolSupplier(str(sdf_path), removeHs=False)
    for mol_index, mol in enumerate(supplier):
        if mol is None:
            continue
        if not mol.HasProp("s_lp_Variant"):
            continue
        if not mol.HasProp("r_i_docking_score"):
            continue
        if not mol.HasProp("s_i_glide_gridfile"):
            continue
        yield mol_index, mol

def _read_sdf_rows(sdf_path: Path) -> list[dict[str, object]]:
    """Load docked SDF molecules with normalized metadata."""
    rows = []
    resolved_sdf = str(sdf_path.resolve())
    for mol_index, mol in _iter_ligand_mols(sdf_path):
        rows.append(
            {
                "mol_index": mol_index,
                "mol": mol,
                "ligand": _parse_variant_ligand(mol.GetProp("s_lp_Variant")),
                "grid": mol.GetProp("s_i_glide_gridfile").strip(),
                "glide_score": float(mol.GetProp("r_i_docking_score")),
                "source_sdf": resolved_sdf,
            }
        )
    return rows

def rdmol_to_pyg_with_pos(mol):
    """Convert an RDKit molecule into a PyG graph with 3D coordinates."""
    data = from_rdmol(mol)
    conf = mol.GetConformer()
    pos = conf.GetPositions()
    data.pos = torch.tensor(pos, dtype=torch.float)
    data.is_protein_atom = torch.zeros(data.num_nodes, dtype=torch.bool)
    return data

In [3]:
rows = _read_sdf_rows(Path("tests/7BU7_P08588_docked/batch_1_docked.sdf"))
graph = rdmol_to_pyg_with_pos(rows[0]["mol"])
graph

Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58])

In [4]:
# prev = rows[0]["mol"].GetNumAtoms()
# for i in range(1000):
#     curr = rows[i]["mol"].GetNumAtoms()
#     if curr != prev:
#         print(i, curr)
#     prev = curr

# ok so we found that at i = 59 prev != curr. There's other examples but this is the first one. 
# our toy dataset will just have 2 graphs. i = 59 and i = 58. 
print(rows[58]["mol"].GetNumAtoms())
print(rows[59]["mol"].GetNumAtoms())

58
57


In [5]:
g1 = rdmol_to_pyg_with_pos(rows[0]["mol"])
g2 = rdmol_to_pyg_with_pos(rows[1]["mol"])

import numpy as np
g1.y = torch.tensor([-10.0])
g2.y = torch.tensor([-1.0])

In [6]:
print(  torch.all((g1.edge_index == g2.edge_index)),
        torch.all((g1.edge_attr == g2.edge_attr)),
        torch.all((g1.pos == g2.pos)),
        torch.all((g1.x == g2.x))
)


tensor(True) tensor(True) tensor(False) tensor(True)


In [7]:
data_list = [g1, g2]
train_loader = DataLoader(data_list, batch_size=2)
val_loader = DataLoader(data_list, batch_size=1)

# Graph Neural Network

In [ ]:
# use edge features. WIP
class GCNPoseRegressor(nn.Module):
    def __init__(self, 
                per_atom_dim, 
                per_bond_dim,
                hidden_dim=128,
                num_layers=2):
        super().__init__()

        total_node_dim = per_atom_dim + 3 # atom features, coordinates. Dont't use is_protein_atom for now. We're not using protein graph embeddings. 

        self.conv1 = GATConv(                
                in_channels=per_atom_dim, 
                out_channels=hidden_dim, 
                heads=2, # attention heads
                concat=True, # concat the attention heads
                # negative_slope: float = 0.2, # leaky relu negative slope. 
                # dropout: float = 0.0, # dropout rate — default is 0
                # add_self_loops: bool = True, 
                edge_dim=per_bond_dim
                # fill_value = 'mean', # default is mean for edge attribute of self-loops. Idk if I need to exp with this yet.
                # bias: bool = True, 
                # residual: bool = False # whether or not to add a residual connection. Seems useful but default is False.
                )
        
        self.conv2 = GATConv(
                hidden_dim, 
                hidden_dim,
                heads=2,
                concat=True,
                edge_dim=hidden_dim
                )
        
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, batch):
        x, edge_index, batch_index = batch.x.float(), batch.edge_index, batch.batch
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch_index)
        x = self.regressor(x)
        return x
    

In [ ]:
# don't use edge features for now. 
class poseGraphNodeRegressor(nn.Module):
    def __init__(self, 
                per_atom_dim, 
                hidden_dim=128):
        
        super().__init__()
    
        per_atom_dim = per_atom_dim + 3 # atom features, coordinates. 

        self.conv1 = GCNConv(in_channels=per_atom_dim,out_channels=hidden_dim)

        self.conv2 = GCNConv(in_channels=hidden_dim,out_channels=hidden_dim)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, batch):
        x, coords, edge_index, batch_index = batch.x.float(), batch.pos, batch.edge_index, batch.batch
        x = torch.cat([x, coords], dim=1)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch_index)
        x = self.regressor(x)
        return x
        

In [ ]:
# per_atom_dim = train_loader.dataset[0].x.shape[1]
# model = poseGraphNodeRegressor(per_atom_dim=per_atom_dim, hidden_dim=128)
# for data in train_loader:
#     print(model(data))

# Train and Evaluate

In [ ]:
def train_epoch(model, loader, optimizer, device, epoch):
    model.train()

    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)
        y = data.y.view_as(out).float()
        loss = F.mse_loss(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    print(f"Epoch {epoch}: Train Loss Item: {loss.item()}, Train Loss Total: {total_loss / len(loader.dataset)}")
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device, epoch):
    model.eval()

    total_loss = 0

    for data in loader:
        data = data.to(device)

        out = model(data)
        y = data.y.view_as(out).float()
        loss = F.mse_loss(out, y)

        total_loss += loss.item() * data.num_graphs

    print(f"Epoch {epoch}: Val Loss Item: {loss.item()}, Val Loss Total: {total_loss / len(loader.dataset)}")
    return total_loss / len(loader.dataset)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_kwargs = {'dropout': 0.5}
per_atom_dim = train_loader.dataset[0].x.shape[1]
# per_bond_dim = train_loader.dataset[0].edge_attr.shape[1]
model = poseGraphNodeRegressor(per_atom_dim=per_atom_dim, hidden_dim=128).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20,
                            min_lr=0.00001)



In [ ]:
for epoch in range(200):
    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        device,
        epoch
    )

    val_loss = evaluate(
        model,
        val_loader,
        device,
        epoch
    )

    print(
        f"Epoch {epoch}: "
        f"train={train_loss:.4f}, "
        f"val={val_loss:.4f}"
    )

In [ ]:
# g1_batched = g1.clone()
# g1_batched.batch = torch.zeros(g1.num_nodes, dtype=torch.long)
# model.eval()
# with torch.no_grad():
#     pred = model(g1_batched.to(device))

model.eval()
model(g1)

# Lightning

In [19]:
# define the LightningModule
class LitPoseGNN(L.LightningModule):
    def __init__(self, num_atom_features, hidden_dim):
        super().__init__()
        num_atom_features = num_atom_features + 3 # atom features, coordinates. 
        self.conv1 = GCNConv(in_channels=num_atom_features,out_channels=hidden_dim)
        self.conv2 = GCNConv(in_channels=hidden_dim,out_channels=hidden_dim)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, batch):
        x, coords, edge_index, batch_idx = batch.x.float(), batch.pos, batch.edge_index, batch.batch
        # coords = torch.zeros_like(coords) # no coordinates for now. coords negative control
        x = torch.cat([x, coords], dim=1)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch_idx)
        x = self.regressor(x)
        return x

    def training_step(self, batch):
        # training_step defines the train loop.
        # it is independent of forward
        x = self.forward(batch)
        y = batch.y.view_as(x).float()
        loss = F.mse_loss(x, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch):
        x = self.forward(batch)
        y = batch.y.view_as(x).float()
        loss = F.mse_loss(x, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss
    

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [20]:
train_loader = DataLoader(data_list, batch_size=1)
val_loader = DataLoader(data_list, batch_size=1)

num_atom_features = train_loader.dataset[0].x.shape[1]
model = LitPoseGNN(num_atom_features=num_atom_features, hidden_dim=128)


checkpoint_callback = L.pytorch.callbacks.ModelCheckpoint(
    save_top_k=1,
    dirpath=Path("runs/pyg_pose_scratch/checkpoints"),
    filename="best",
    monitor="val_loss",
    mode="min",
    save_last=True,
    enable_version_counter=False
)

loss_history = EpochLossHistoryCallback()

trainer = L.Trainer(    logger=False,
                        enable_checkpointing=True,
                        enable_progress_bar=True,
                        accelerator="auto",
                        max_epochs=200,
                        default_root_dir=Path("runs/pyg_pose_scratch"),
                        callbacks=[
                            checkpoint_callback,
                            loss_history.bind_lightning_callback()
                        ]
                    )

trainer.fit(        model=model, 
                    train_dataloaders=train_loader,
                    val_dataloaders=val_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/zanechan/anaconda3/envs/pyg_pose/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/zanechan/boltzProject/thndr_pose/runs/pyg_pose_scratch/checkpoints exists and is not empty.

  | Name      | Type       | Params | Mode  | FLOPs
---------------------------------------------------------
0 | conv1     | GCNConv    | 1.7 K  | train | 0    
1 | conv2     | GCNConv    | 16.5 K | train | 0    
2 | regressor | Sequential | 16.6 K | train | 0    
---------------------------------------------------------
34.8 K    Trainable params
0         Non-trainable params
34.8 K    Total params
0.139     Total estimated model params size

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/zanechan/anaconda3/envs/pyg_pose/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/zanechan/anaconda3/envs/pyg_pose/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/Users/zanechan/anaconda3/envs/pyg_pose/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 199: 100%|██████████| 2/2 [00:00<00:00, 46.79it/s, val_loss=1.66e-11, train_loss=3.01e-11] 

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199: 100%|██████████| 2/2 [00:00<00:00, 45.13it/s, val_loss=1.66e-11, train_loss=3.01e-11]


In [21]:
model(g2)

tensor([[-1.0000]], grad_fn=<AddmmBackward0>)

In [22]:
g1, g2

(Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58], y=[1]),
 Data(x=[58, 9], edge_index=[2, 120], edge_attr=[120, 3], pos=[58, 3], is_protein_atom=[58], y=[1]))

In [23]:
# load checkpoint
checkpoint = "./runs/pyg_pose_scratch/checkpoints/last.ckpt"
device = torch.device('cpu')
poseGNN = LitPoseGNN.load_from_checkpoint(checkpoint, num_atom_features=num_atom_features, hidden_dim=128).to(device)

poseGNN(g2)

tensor([[-1.0000]], grad_fn=<AddmmBackward0>)